# CFM56-5B — Gázkar Szimulátor
**Interaktív tolóerő-szabályozás** — A csúszka a T4 turbinabemeneti hőmérsékletet állítja be, a legördülő menü a repülési fázist.

- **0 %** → alapjárat (~1000 K)
- **100 %** → teljes tolóerő (~1700 K)

In [1]:
import sys
sys.path.insert(0, '..')

import importlib
import visualization.station_diagram as sd
import visualization.ts_diagram as ts
import visualization.model_3d as m3d
importlib.reload(sd)
importlib.reload(ts)
importlib.reload(m3d)

from engine import run_design_point
from visualization.station_diagram import plot_station_diagram
from visualization.ts_diagram import plot_ts_diagram
from visualization.model_3d import plot_3d_model
from IPython.display import display
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
FLIGHT_PHASES = {
    'Felszállás (0 ft, Mach 0.25)':        {'altitude_ft': 0,     'mach': 0.25, 'key': 'takeoff'},
    'Emelkedés (15 000 ft, Mach 0.50)':    {'altitude_ft': 15000, 'mach': 0.50, 'key': 'climb'},
    'Utazórepülés (35 000 ft, Mach 0.78)': {'altitude_ft': 35000, 'mach': 0.78, 'key': 'cruise'},
}

throttle_slider = widgets.IntSlider(
    value=100, min=0, max=100, step=5,
    description='Gázkar [%]:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='420px')
)

phase_dropdown = widgets.Dropdown(
    options=list(FLIGHT_PHASES.keys()),
    value=list(FLIGHT_PHASES.keys())[0],
    description='Repülési fázis:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='420px')
)

controls = widgets.HBox([phase_dropdown, throttle_slider])
output = widgets.Output()

def update(change):
    pct = throttle_slider.value / 100.0
    T4  = 1000.0 + pct * 700.0
    fp  = FLIGHT_PHASES[phase_dropdown.value]

    result = run_design_point(
        flight_phase=fp['key'],
        altitude_ft=fp['altitude_ft'],
        mach=fp['mach'],
        T4_override=T4
    )

    with output:
        output.clear_output(wait=True)
        print(f"Fázis: {phase_dropdown.value}")
        print(f"Gázkar: {throttle_slider.value} %   |   T4 = {T4:.0f} K")
        print('─' * 55)
        result.summary()

        fig1 = plot_station_diagram(result)
        plt.tight_layout()
        display(fig1)
        plt.close(fig1)

        fig2 = plot_ts_diagram([result])
        plt.tight_layout()
        display(fig2)
        plt.close(fig2)

        fig3 = plot_3d_model(result)
        display(fig3)

throttle_slider.observe(update, names='value')
phase_dropdown.observe(update, names='value')

display(controls, output)
update(None)